# Task 2 — Generate Product Descriptions

Using the rubric defined in Task 1, this notebook:
1. Defines a system prompt that instructs the model to produce a persuasive **50–90 word** product description.
2. Calls the Nebius API for every product in the dataset.
3. Collects `generated_description`, `latency_ms`, `input_tokens`, and `output_tokens` for each call.
4. Saves everything to `assignment_01.xlsx` with blank rubric columns ready for Task 3.

**Model:** `meta-llama/Meta-Llama-3.1-8B-Instruct` (Nebius Token Factory)

## Configuration

In [1]:
import time
import os
import pandas as pd
from openai import OpenAI

API_KEY  = os.getenv("NEBIUS_API_KEY")
BASE_URL = "https://api.tokenfactory.nebius.com/v1/"
MODEL    = "meta-llama/Meta-Llama-3.1-8B-Instruct"

DATASET_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "Assignment_01_product_dataset.xlsx")
OUTPUT_PATH  = os.path.join(os.path.dirname(os.path.abspath("__file__")), "assignment_01.xlsx")

## System Prompt

The prompt follows the rubric criteria directly:
- **Length** constraint is stated explicitly (50–90 words).
- **Tone** is described as warm, benefit-focused — not a spec list.
- **Grounding** rule forbids inventing any information not in the product data.
- **Grammar** is called out explicitly.
- Output format is constrained: description only, no commentary.

In [2]:
SYSTEM_PROMPT = """You are an expert e-commerce copywriter. Your task is to write a persuasive product description.

Rules you must follow:
1. Length: write between 50 and 90 words — no more, no less.
2. Tone: use a warm, confident, benefit-focused sales voice. Avoid dry spec lists and excessive hype.
3. Grounding: base every claim strictly on the product information provided. Do not invent features, materials, or specifications that are not mentioned.
4. Grammar: use correct spelling, punctuation, and sentence structure throughout.
5. Output: return only the product description — no headings, no labels, no extra commentary.
"""

## `build_user_message(row)`

Constructs the user-turn message for a single product by formatting the four available product fields into a structured block:

- **`product_name`** — gives the model the product identity so it can write a title-appropriate description.
- **`Product_attribute_list`** — the primary source of factual claims (specs, features, connectivity, etc.). The model must stay grounded to this list.
- **`material`** — physical composition of the product. Included explicitly because material claims are a common grounding failure: models tend to invent materials (e.g., "stainless steel" for a plastic product) when this field is absent.
- **`warranty`** — a trust signal customers look for. Included so the model can mention it if it fits naturally, without having to invent a warranty period.

The closing instruction repeats the 50–90 word constraint in the user turn even though it already appears in the system prompt. This deliberate redundancy reinforces the length requirement: the system prompt sets the overall persona and rules, but models respond more reliably to constraints that appear close to the generation request.

In [3]:
def build_user_message(row: pd.Series) -> str:
    return (
        f"Product name: {row['product_name']}\n"
        f"Attributes: {row['Product_attribute_list']}\n"
        f"Material: {row['material']}\n"
        f"Warranty: {row['warranty']}\n\n"
        "Write a persuasive 50\u201390 word product description based on the information above."
    )

## `generate_description(client, row)`

Makes a single chat completion call for one product row and returns a flat dictionary with four fields:

| Field | Description |
|---|---|
| `generated_description` | The model's output, stripped of leading/trailing whitespace |
| `latency_ms` | End-to-end wall-clock time in milliseconds, measured around the API call |
| `input_tokens` | Prompt token count reported by the API (`usage.prompt_tokens`) |
| `output_tokens` | Completion token count reported by the API (`usage.completion_tokens`) |

**Parameter choices:**

- **`temperature=0.7`** — a moderate value that produces natural, varied language while remaining coherent. A lower temperature (e.g., 0.1) would make descriptions repetitive across similar products; a higher value (e.g., 1.0) risks incoherent sentences or hallucinated facts.
- **`max_tokens=200`** — set well above the 90-word ceiling (~130 tokens) to give the model room to complete a sentence naturally, without being cut off mid-description. The system prompt enforces the 90-word limit; `max_tokens` is a safety cap, not the primary length control.

**Latency measurement** wraps only the API call itself (`time.time()` before and after `client.chat.completions.create`), so it captures network round-trip and model inference time but excludes data loading and Excel I/O.

**Error handling** is intentionally left to the caller (the run loop below), which catches exceptions and writes sentinel values (`-1`) so that a single failure does not abort the entire batch.

In [4]:
def generate_description(client: OpenAI, row: pd.Series) -> dict:
    user_msg = build_user_message(row)

    start = time.time()
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_msg},
        ],
        temperature=0.7,
        max_tokens=200,
    )
    latency_ms = round((time.time() - start) * 1000)

    return {
        "generated_description": response.choices[0].message.content.strip(),
        "latency_ms":            latency_ms,
        "input_tokens":          response.usage.prompt_tokens,
        "output_tokens":         response.usage.completion_tokens,
    }

## Run - Generate Descriptions for All Products

Iterates over all 50 rows sequentially (not in parallel) for two reasons: the Nebius API may rate-limit concurrent requests, and sequential calls make latency measurements per product meaningful. Progress is printed inline so failures are visible immediately without waiting for the full batch to complete.

On success, word count and latency are printed as a quick sanity check - if the model repeatedly returns descriptions well outside 50–90 words, the system prompt needs adjustment. On failure, sentinel values (`-1`) are recorded so the row is preserved in the output file and the issue can be investigated in Task 3.

In [5]:
# Load dataset
df = pd.read_excel(DATASET_PATH)
print(f"Loaded {len(df)} products.")

# Init client
client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

# Generate descriptions
results = []
for idx, row in df.iterrows():
    print(f"  [{idx+1:02d}/{len(df)}] {row['product_name']} ...", end=" ", flush=True)
    try:
        result = generate_description(client, row)
        word_count = len(result["generated_description"].split())
        print(f"OK  ({word_count} words, {result['latency_ms']} ms)")
    except Exception as e:
        print(f"ERROR: {e}")
        result = {
            "generated_description": "",
            "latency_ms":            -1,
            "input_tokens":          -1,
            "output_tokens":         -1,
        }
    results.append(result)

Loaded 50 products.
  [01/50] Apple iPhone 15 Pro ... OK  (77 words, 3453 ms)
  [02/50] Samsung Galaxy S24 Ultra ... OK  (84 words, 2532 ms)
  [03/50] Google Pixel 8 Pro ... OK  (75 words, 2556 ms)
  [04/50] Sony WH‑1000XM5 Headphones ... OK  (70 words, 2050 ms)
  [05/50] Bose QuietComfort Ultra Earbuds ... OK  (74 words, 2342 ms)
  [06/50] Amazon Echo Dot (5th Gen) ... OK  (78 words, 2218 ms)
  [07/50] Dell XPS 13 9310 Laptop ... OK  (81 words, 2550 ms)
  [08/50] Apple MacBook Air 13″ (M3) ... OK  (71 words, 2389 ms)
  [09/50] Microsoft Surface Pro 10 ... OK  (71 words, 2203 ms)
  [10/50] Garmin Forerunner 255 ... OK  (81 words, 3128 ms)
  [11/50] Fitbit Charge 6 ... OK  (80 words, 2555 ms)
  [12/50] GoPro HERO12 Black ... OK  (69 words, 2239 ms)
  [13/50] DJI Mini 4 Pro Drone ... OK  (80 words, 3272 ms)
  [14/50] Nintendo Switch OLED ... OK  (71 words, 2172 ms)
  [15/50] PlayStation 5 Slim ... OK  (83 words, 2237 ms)
  [16/50] Xbox Series X ... OK  (73 words, 2134 ms)
  [17/50] Insta

## Save to Excel

The results DataFrame is joined column-wise to the original dataset (preserving all source columns) and written to `assignment_01.xlsx`. Seven blank rubric columns are added, one per criterion, along with a blank `final_score` column. These columns are left empty intentionally: they will be filled in during Task 3 (manual evaluation) and Task 5 (automated judge), keeping the output file as the single source of truth across tasks.

In [6]:
results_df = pd.DataFrame(results)
output_df  = pd.concat([df.reset_index(drop=True), results_df], axis=1)

# Blank rubric columns (filled in Task 3)
rubric_criteria = ["fluency", "grammar", "tone", "length", "grounding", "latency", "cost"]
for criterion in rubric_criteria:
    output_df[criterion] = ""
output_df["final_score"] = ""

output_df.to_excel(OUTPUT_PATH, index=False)
print(f"Saved {len(output_df)} rows to: {OUTPUT_PATH}")

Saved 50 rows to: C:\Users\hayla\Desktop\Nebius_Performence_AI\HW1\Tasks\assignment_01.xlsx


## Run Summary

In [7]:
successful = results_df[results_df["latency_ms"] > 0]
if len(successful):
    print(f"--- Run summary ---")
    print(f"  Successful calls : {len(successful)} / {len(df)}")
    print(f"  Avg latency      : {successful['latency_ms'].mean():.0f} ms")
    print(f"  Avg input tokens : {successful['input_tokens'].mean():.1f}")
    print(f"  Avg output tokens: {successful['output_tokens'].mean():.1f}")

--- Run summary ---
  Successful calls : 50 / 50
  Avg latency      : 2390 ms
  Avg input tokens : 212.9
  Avg output tokens: 102.0
